In [ ]:
!apt-get -qq install aria2 > /dev/null 2>&1
!pip install --quiet omnicloudmask==1.7.0

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

In [ ]:
from pathlib import Path

import pandas as pd

from cloudband.acquisition import manifest, zenodo
from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.labels import pixbox
from cloudband.pipelines import pixbox_s2
from cloudband.eval.report import as_percentages, to_frame

reload_package("cloudband")

In [ ]:
def report(position, total):
    print(f"{position}/{total}", flush=True)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/data/pixbox")
SCENES_DIR = Path("/content/data/scenes")
PRED_DIR = Path("/content/data/predictions")
RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)

In [ ]:
zenodo.download(zenodo.LABELS, DATA_DIR)
!unzip -o -q {DATA_DIR}/{zenodo.LABELS_ARCHIVE} -d {DATA_DIR}

table = pixbox_s2.load_reference(DATA_DIR / zenodo.LABELS_CSV)
print("scorable pixels:", len(table))
print("dropped:", pixbox.unavailable_pixel_counts(pd.read_csv(DATA_DIR / zenodo.LABELS_CSV)))

In [ ]:
archive = zenodo.download(zenodo.SCENES, Path("/content/data"))
extracted = zenodo.extract_scenes(archive, SCENES_DIR)
print("extracted:", len(extracted))

In [ ]:
expected = [
    name.replace(".SAFE", "")
    for pid, name in pixbox.PRODUCT_ID_TO_SCENE.items()
    if pid not in pixbox.UNAVAILABLE_PRODUCT_IDS
]
statuses = manifest.check_collection(SCENES_DIR, expected)
print(manifest.summarise(statuses))
manifest.require_complete(statuses)

In [ ]:
config = ocm.InferenceConfig(model_version=ocm.LATEST_MODEL_VERSION)
scene_paths = [status.path for status in statuses]

masks = ocm.predict_scenes(scene_paths, PRED_DIR, config)
print("masks:", len(masks))

In [ ]:
scored = pixbox_s2.attach_predictions(table, PRED_DIR)
confusions = pixbox_s2.score(scored)
as_percentages(to_frame(confusions))